In [ ]:
import os
import shutil
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms

import medicalDataLoader_Enes as medicalDataLoader
from utils_Enes import to_var, getTargetSegmentation, dice_per_class, hausdorff_per_class, keep_largest_component
from UNet_Base_Enes import UNet

In [ ]:
MODEL_PATHS = os.environ.get('MODELS', 'models/mean_teacher_w10_seed0/last.pth,models/mean_teacher_w10_seed1/last.pth').split(',')
TEST_DIR = os.environ.get('TEST_DIR', '')
OUT_DIR = 'predictions'
BATCH_SIZE = 8

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../Data/'

In [ ]:
def load_models(paths):
    models = []
    for path in paths:
        net = UNet(num_classes=4)
        net.load_state_dict(torch.load(path, map_location='cpu'))
        if torch.cuda.is_available():
            net.cuda()
        net.eval()
        models.append(net)
    return models


@torch.no_grad()
def predict(models, images):
    images = to_var(images)
    probs = torch.stack([torch.softmax(net(images), dim=1) for net in models]).mean(dim=0)
    return probs.argmax(dim=1).cpu()


def evaluate(models, loader):
    preds, targets = [], []
    for images, labels, _ in loader:
        preds.append(predict(models, images))
        targets.append(getTargetSegmentation(labels))
    preds = keep_largest_component(torch.cat(preds))
    targets = torch.cat(targets)
    return dice_per_class(preds, targets), hausdorff_per_class(preds, targets)


def print_scores(name, dsc, hd):
    print(name)
    print('   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*dsc, np.mean(dsc)))
    print('   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*hd))


def save_predictions(models, img_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    to_tensor = transforms.ToTensor()
    names = sorted(f for f in os.listdir(img_dir) if f.endswith('.png'))
    for k in range(0, len(names), BATCH_SIZE):
        batch = names[k:k + BATCH_SIZE]
        images = torch.stack([to_tensor(Image.open(os.path.join(img_dir, n)).convert('L')) for n in batch])
        preds = keep_largest_component(predict(models, images))
        for name, pred in zip(batch, preds):
            Image.fromarray((pred.numpy() * 85).astype(np.uint8)).save(os.path.join(out_dir, name))
    print('{} masques enregistrés dans {}'.format(len(names), out_dir))

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
val_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform, mask_transform=transform)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

models = load_models(MODEL_PATHS)
for path, net in zip(MODEL_PATHS, models):
    print_scores(path, *evaluate([net], val_loader))
if len(models) > 1:
    print_scores('Ensemble de {} modèles'.format(len(models)), *evaluate(models, val_loader))

In [ ]:
save_predictions(models, os.path.join(DATA_DIR, 'val', 'Img'), os.path.join(OUT_DIR, 'val'))
check = np.array(Image.open(os.path.join(OUT_DIR, 'val', sorted(os.listdir(os.path.join(OUT_DIR, 'val')))[0])))
print('Valeurs dans un masque prédit :', np.unique(check))

if TEST_DIR:
    save_predictions(models, TEST_DIR, os.path.join(OUT_DIR, 'test'))
    shutil.make_archive('predictions_test', 'zip', os.path.join(OUT_DIR, 'test'))
    print('Archive : predictions_test.zip')